# AI AINAK — Notebook 3: Kinematic Gesture Recognition Module

**Branch 3:** MediaPipe Pose 33-Point Skeletal Keypoint Extraction, Torso-Centered Normalization, 24-Frame Temporal Buffer, and 2-Layer GestureLSTM Inference.

This notebook executes the Gesture recognition sensory branch of AI AINAK:
1. Loads the **Google MediaPipe PoseLandmarker** task bundle (`pose_landmarker_lite.task`).
2. Loads the trained **2-layer unidirectional GestureLSTM** checkpoint (`gesture_lstm.pt`).
3. Ingests video frames from `datasets/recordings/session-multimodal-01/video.mp4`.
4. Extracts 33 3D skeletal landmarks $(x, y, z, 	ext{visibility})$ per frame.
5. Applies torso-relative normalization ($33 	imes 4 = 132$ dimensions).
6. Maintains a sliding FIFO 24-frame temporal buffer (~0.8s).
7. Classifies bodily gestures: `pointing`, `waving`, `stop`, `neutral`.
8. Visualizes the 33-point skeletal bones overlaid on real video frames.
9. Extracts the **128-dimensional hidden state embedding** from GestureLSTM.
10. Saves formatted outputs to `gesture_outputs.npz` for multimodal Transformer fusion.


In [ ]:
# Step 1: Install Dependencies
!pip install -q mediapipe opencv-python torch numpy matplotlib


In [ ]:
# Environment Detection & Path Resolution
import os, sys, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    # Mount Google Drive if available
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    # Locate project root
    drive_candidates = [
        Path("/content/drive/MyDrive/AINAK_Colab"),
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), Path.cwd())
else:
    print("Running in local environment.")
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root resolved to: {PROJECT_ROOT}")

# Helper to find checkpoint across standard folders
def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Checkpoint not found: {filename}. Checked: {[str(c) for c in candidates]}")

# Helper to find dataset / recording session
def find_session_media(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Media file {filename} for session {session_id} not found.")

# Ensure output directories exist
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)



## Checkpoint & Input Ingestion
Locate MediaPipe pose task model, GestureLSTM weights, and the session video.


In [ ]:
import cv2
import torch
from torch import nn
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import time

pose_task_path = find_checkpoint("pose_landmarker_lite.task", "gesture")
lstm_path = find_checkpoint("gesture_lstm.pt", "gesture")
video_path = find_session_media("session-multimodal-01", "video.mp4")

print(f"Loaded MediaPipe Pose Landmarker: {pose_task_path} ({pose_task_path.stat().st_size / 1e6:.2f} MB)")
print(f"Loaded GestureLSTM Checkpoint: {lstm_path} ({lstm_path.stat().st_size / 1e6:.2f} MB)")
print(f"Selected Video Session: {video_path}")


## MediaPipe Pose Tasks API & Normalization Implementation
Defines landmark normalization and 2-layer GestureLSTM architecture matching `ainak.models.gesture`.


In [ ]:
import mediapipe as mp

GESTURE_LABELS = ("pointing", "waving", "stop", "neutral")

# 1. Landmark Normalizer (33 keypoints -> 132-D vector)
def normalize_landmarks(landmarks_33x4: np.ndarray) -> np.ndarray:
    p = np.asarray(landmarks_33x4, dtype=np.float32).reshape(33, 4).copy()
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    # MediaPipe landmarks: Left shoulder=11, Right shoulder=12, Left hip=23, Right hip=24
    center = (p[23, :3] + p[24, :3]) / 2.0
    scale = max(float(np.linalg.norm(p[11, :2] - p[12, :2])), 1e-3)
    p[:, :3] = (p[:, :3] - center) / scale
    p[p[:, 3] < 0.2, :3] = 0.0
    p[:, :3] = np.clip(p[:, :3], -5.0, 5.0)
    p[:, 3] = np.clip(p[:, 3], 0.0, 1.0)
    return p.reshape(-1)

# 2. GestureLSTM Architecture (2-layer unidirectional)
class GestureLSTM(nn.Module):
    def __init__(self, feature_dim: int = 132, hidden_size: int = 128, classes: int = 4, dropout: float = 0.2, layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(feature_dim, hidden_size, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, classes)

    def forward(self, x: torch.Tensor):
        _, (h, _) = self.lstm(x)
        embedding = self.norm(h[-1])
        return self.classifier(embedding), embedding

# Load GestureLSTM model
lstm_payload = torch.load(str(lstm_path), map_location="cpu", weights_only=False)
gesture_model = GestureLSTM(feature_dim=132, hidden_size=128, classes=4, layers=2)
gesture_model.load_state_dict(lstm_payload["state_dict"])
gesture_model.eval()

# 3. MediaPipe PoseLandmarker Tasks API Wrapper
base_options = mp.tasks.BaseOptions(model_asset_path=str(pose_task_path))
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(options)

print("PoseLandmarker Tasks API & GestureLSTM initialized successfully.")


## Process Video Frames, Extract Skeletons & Run GestureLSTM
We feed video frames to MediaPipe, buffer 24 frames, and run GestureLSTM forward passes.


In [ ]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

buffer = deque(maxlen=24)
timestamps = []
features_128d = []
predicted_classes = []
confidences = []
landmarks_history = []
latencies_ms = []
sample_skeleton_frames = []

frame_idx = 0
print("Extracting pose landmarks and running GestureLSTM...")
t0 = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    ts = frame_idx / fps
    frame_ms = int(ts * 1000)
    frame_idx += 1

    t_start = time.perf_counter()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, frame_ms)

    if result.pose_landmarks and len(result.pose_landmarks) > 0:
        lm_list = result.pose_landmarks[0]
        points = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in lm_list], dtype=np.float32)
        norm_132 = normalize_landmarks(points)
        buffer.append(norm_132)

        # Run LSTM if buffer full (24 frames)
        if len(buffer) == 24:
            seq_array = np.stack(buffer)[None, :]
            with torch.no_grad():
                x_tensor = torch.from_numpy(seq_array)
                logits, emb_128 = gesture_model(x_tensor)
                probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
            
            top_idx = int(probs.argmax())
            top_cls = GESTURE_LABELS[top_idx]
            top_conf = float(probs[top_idx])
            emb_vector = emb_128[0].cpu().numpy()
        else:
            top_cls = "buffering"
            top_conf = 0.0
            emb_vector = np.zeros(128, dtype=np.float32)

        landmarks_history.append(points)
    else:
        norm_132 = np.zeros(132, dtype=np.float32)
        emb_vector = np.zeros(128, dtype=np.float32)
        top_cls = "neutral"
        top_conf = 0.0
        landmarks_history.append(np.zeros((33, 4), dtype=np.float32))

    t_latency = (time.perf_counter() - t_start) * 1000

    timestamps.append(ts)
    features_128d.append(emb_vector)
    predicted_classes.append(top_cls)
    confidences.append(top_conf)
    latencies_ms.append(t_latency)

    if frame_idx % 25 == 0 or frame_idx == 1:
        sample_skeleton_frames.append((ts, rgb.copy(), landmarks_history[-1], top_cls, top_conf))

cap.release()
print(f"Completed gesture extraction: {len(timestamps)} frames processed in {time.time() - t0:.2f}s.")
print(f"Mean Gesture Latency: {np.mean(latencies_ms):.2f} ms | p95 Latency: {np.percentile(latencies_ms, 95):.2f} ms")


## Skeletal Overlay Visualization


In [ ]:
POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (24, 26), (25, 27), (26, 28),
    (27, 29), (28, 30), (29, 31), (30, 32),
    (0, 11), (0, 12)
]

fig, axes = plt.subplots(1, min(3, len(sample_skeleton_frames)), figsize=(16, 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for idx, (ts, rgb_frame, lms, cls_name, conf) in enumerate(sample_skeleton_frames[:3]):
    h, w = rgb_frame.shape[:2]
    annotated = rgb_frame.copy()

    # Draw skeletal bones
    if lms.any():
        for i, j in POSE_CONNECTIONS:
            p1, p2 = lms[i], lms[j]
            if p1[3] > 0.3 and p2[3] > 0.3:
                cv2.line(annotated, (int(p1[0]*w), int(p1[1]*h)), (int(p2[0]*w), int(p2[1]*h)), (16, 185, 129), 3)
        for pt in lms:
            if pt[3] > 0.3:
                cv2.circle(annotated, (int(pt[0]*w), int(pt[1]*h)), 4, (245, 158, 11), -1)

    axes[idx].imshow(annotated)
    axes[idx].set_title(f"Time: {ts:.2f}s | Gesture: {cls_name.upper()} ({conf*100:.0f}%)", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


## Export Gesture Outputs to `gesture_outputs.npz`


In [ ]:
gesture_out_path = OUTPUTS_DIR / "gesture_outputs.npz"

np.savez_compressed(
    str(gesture_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(features_128d, dtype=np.float32),
    landmarks=np.array(landmarks_history, dtype=np.float32),
    predicted_classes=np.array(predicted_classes),
    confidences=np.array(confidences, dtype=np.float32),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported gesture outputs to: {gesture_out_path}")
print(f"File size: {gesture_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(features_128d).shape} (Matches 128-D Transformer contract)")
